
<div align="center">
  <h1></h1>
  <h1>⚒️ Mistral 7B Fine-Tuning using PEFT</h1>
  <h4 align="center">!!!!! This session will be recorded !!!!!</h4>
</div>

In this exercise, we will fine-tune the Mistral 7B model from HuggingFace 🤗 using a parameter-efficient technique (PEFT) known as LoRA (Low-Rank Adaptation). This approach allows us to adapt the model for a specific task—Dialogue Summarization—while keeping computational costs low. We willl demonstrate how to apply LoRA and then evaluate the model’s performance before and after fine-tuning to highlight the improvements.

Tools Required:
*  [HuggingFace 🤗](https://huggingface.co/)
*  [BitsandBytes](https://huggingface.co/docs/bitsandbytes/main/en/index)
*  [PEFT](https://huggingface.co/docs/peft/index)
*  [TRL](https://huggingface.co/docs/trl/index)
*  [Accelerate](https://huggingface.co/docs/accelerate/en/index)

Learning Outcomes:

1. **Load LLMs Locally:** Learn to load large models from HuggingFace with sufficient resources.
2. **Use HuggingFace APIs:** Understand and navigate HuggingFace’s Dataset and Training APIs.
3. **Work with Quantized Models:** Load and use quantized models for resource-limited environments.
4. **Prepare Instruction Datasets:** Learn how instruction datasets are created for fine-tuning.
5. **Efficient Fine-Tuning (LoRA):** Apply LoRA for parameter-efficient fine-tuning with minimal accuracy loss.
6. **Understand Hyperparameters:** Gain insight into training hyperparameters and their effects.
7. **Save and Load Models:** Become familiar with saving and loading fine-tuned models.

Exercise Steps
 * 🔮 Loading Mistral Model
 * 🔮 LoRA Fineuning
 * 🔮 Test Tuned Model

For this exercise, we will need to use a GPU instead of a CPU.

To enable GPU on Colab:  
1. Go to **Runtime** -> **Change Runtime Type**.  
2. Select **T4 GPU** and save.  

After changing the runtime, restart the session to apply the changes.

In [ ]:
# Install required libraries
!pip install -q torch                                  # Pytorch
!pip install -q transformers datasets                  # Comes from HuggingFace
!pip install -q bitsandbytes                           # For quantization from HuggingFace
!pip install -q peft                                   # Parameter-efficient Fine-tuning from HuggingFace
!pip install -q trl                                    # For supervised fine-tuning for LLMs from HuggingFace
!pip install -q accelerate                             # For distributed training from HuggingFace

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 480.6/480.6 kB 12.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 116.3/116.3 kB 9.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 179.3/179.3 kB 7.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 134.8/134.8 kB 8.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 194.1/194.1 kB 9.1 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
gcsfs 2024.10.0 requires fsspec==2024.10.0, but you have fsspec 2024.9.0 which is incompatible.
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 122.4/122.4 MB 7.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 310.9/310.9 kB 9.6 MB/s eta 0:00:00


# 🔮 Loading Mistral Model

We will be working with the **7B Mistral model [[mistralai/Mistral-7B-v0.3](https://huggingface.co/mistralai/Mistral-7B-v0.3)] from HuggingFace, in HF format**. According to HuggingFace's [documentation](https://huggingface.co/transformers/v3.5.1/model_doc/auto.html#automodelforcausallm), the recommended class for this model is `AutoModelForCausalLM`.

![](https://i.imgur.com/Tydeaak.png)


The term **causal head** refers to an **autoregressive** model, similar to the transformer decoder we implemented in the previous exercise.

To access the model, you will need to agree to share your contact information:

1. Visit [Mistral 7B v0.3 on HuggingFace](https://huggingface.co/mistralai/Mistral-7B-v0.3) and accept the terms.  
2. Generate a token from [HuggingFace Token Settings](https://huggingface.co/settings/tokens/new?tokenType=read).  
3. Assign the token to the `HF_API_TOKEN` variable in the following code cell:

In [ ]:
# To load the model, we’ll need a HuggingFace API token.
from google.colab import userdata
HF_API_TOKEN = userdata.get('HF_API_TOKEN')

# You can generate your token here: https://huggingface.co/settings/tokens/new?tokenType=read
# Uncomment the following line and replace "YOUR_TOKEN" with your actual token:
# HF_API_TOKEN = "YOUR_TOKEN"

# Set the token as an environment variable
import os
os.environ["HF_TOKEN"] = HF_API_TOKEN

In [ ]:
# Import the required class
from transformers import AutoModelForCausalLM

# Load the full model
model = AutoModelForCausalLM.from_pretrained("mistralai/Mistral-7B-v0.3")

/usr/local/lib/python3.10/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/601 [00:00<?, ?B/s]

model.safetensors.index.json:   0%|          | 0.00/23.9k [00:00<?, ?B/s]

model-00001-of-00003.safetensors:   0%|          | 0.00/4.95G [00:00<?, ?B/s]

model-00002-of-00003.safetensors:   0%|          | 0.00/5.00G [00:00<?, ?B/s]

model-00003-of-00003.safetensors:   0%|          | 0.00/4.55G [00:00<?, ?B/s]

Loading checkpoint shards:   0%|          | 0/3 [00:00<?, ?it/s]

**Boom! GPU Out of Memory!**

Let’s calculate the memory requirement for a 7-billion-parameter model using float32 (4-byte) precision:

$
\text{Memory (GB)} = \frac{7,000,000,000 \times \frac{32}{8}}{1024^3}
$

Simplifying:

$
\text{Memory (GB)} = \frac{7,000,000,000 \times 4}{1024^3}
$

$
\text{Memory (GB)} = \frac{28,000,000,000}{1024^3}
$

$
\text{Memory (GB)} \approx 26.06 \, \text{GB}
$


For training, we need 8 times this amount to account for optimizer states, gradients, and parameters:

$
\text{Memory (GB)} = 26.06 \times 8 \approx 208.48 \, \text{GB}
$

## Model Quantization

To reduce the model's memory footprint, we can decrease the memory usage of each parameter through model quantization techniques.

The following figure illustrates the value range of model parameters based on the data type used for storage.

![](https://substackcdn.com/image/fetch/f_auto,q_auto:good,fl_progressive:steep/https%3A%2F%2Fsubstack-post-media.s3.amazonaws.com%2Fpublic%2Fimages%2F7dbb8398-9f3f-4d9a-b63f-591cb37bdbdd_1144x856.png)

If we have a trained model that was initially trained with high-precision data type as FP32, we can easily convert it to a lower-precision structure using quantization, as shown below:

![](https://substackcdn.com/image/fetch/f_auto,q_auto:good,fl_progressive:steep/https%3A%2F%2Fsubstack-post-media.s3.amazonaws.com%2Fpublic%2Fimages%2Fe9d17077-d9af-4b37-9b9b-57ef9aaa1ca9_680x486.png)

In this exercise, we will quantize the model parameters to fit each parameter into **4 bits** instead of 32 bits by reducing the dynamic range from [$-3.4 \times 10^{38}$, $3.4 \times 10^{38}$] to [-8, 7].

We will apply 4-bit quantization to the model using the `BitsAndBytes` library, as outlined in the [QLoRA](https://arxiv.org/pdf/2305.14314.pdf) paper. This will be applied to all weights and activations in the model.

For more information on model quantization, you can refer to the HuggingFace documentation here:  
[HuggingFace Model Quantization Guide](https://huggingface.co/docs/transformers/main/main_classes/quantization)

In [ ]:
# Load BitsAndBytes object from HuggingFace Transformers
from transformers import BitsAndBytesConfig
import torch

# Set up the quantization configuration
quant_config = BitsAndBytesConfig(
    load_in_4bit=True,                     # Use 4-bit quantization (Q = 4 bits)
    bnb_4bit_use_double_quant=True,        # Double quantization: quantize the quantization constants to save an additional 0.4 bits per parameter
    bnb_4bit_quant_type="nf4",             # Use 4-bit NormalFloat Quantization (optimal for normal weights; enforces w ∈ [-1,1])
    bnb_4bit_compute_dtype=torch.bfloat16  # Dequantize to 16-bits before computation (as described in the paper)
)

# Pass the quantization configuration when loading the model
from transformers import AutoModelForCausalLM

# Load the quantized model
model = AutoModelForCausalLM.from_pretrained("mistralai/Mistral-7B-v0.3",
                                             quantization_config=quant_config)


/usr/local/lib/python3.10/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(
`low_cpu_mem_usage` was None, now default to True since model is quantized.


Loading checkpoint shards:   0%|          | 0/3 [00:00<?, ?it/s]

Let’s calculate the memory requirement for a 7-billion-parameter model using 4-bit (0.5 byte) precision:

$
\text{Memory (GB)} = \frac{7,000,000,000 \times \frac{4}{8}}{1024^3}
$

Simplifying:

$
\text{Memory (GB)} = \frac{7,000,000,000 \times 0.5}{1024^3}
$

$
\text{Memory (GB)} = \frac{3,500,000,000}{1024^3}
$

$
\text{Memory (GB)} \approx 3.26 \, \text{GB} \leq 15 \, \text{GB}
$

## Loading the Tokenizer

Once the model is loaded, we need to use it with a tokenizer that converts our text input into numerical format, which the model can process.

To do this, we will use the model's tokenizer, which is also available on HuggingFace.

In [ ]:
# Get the tokenizer object from HuggingFace
from transformers import AutoTokenizer

# Download and initialize the tokenizer
tokenizer = AutoTokenizer.from_pretrained("mistralai/Mistral-7B-v0.3")

tokenizer_config.json:   0%|          | 0.00/137k [00:00<?, ?B/s]

tokenizer.model:   0%|          | 0.00/587k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.96M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/414 [00:00<?, ?B/s]

In [ ]:
# Set the padding token to be the same as the end-of-sequence token.
# This is a common practice, especially for decoder-only models like Mistral.
# Padding ensures all sequences in a batch are of equal length.
tokenizer.pad_token = tokenizer.eos_token

# Specify that padding should be applied to the right side of the sequences.
# This is the standard behavior for Mistral and many other causal language models.
tokenizer.padding_side = "right"

## Testing the model

Let’s test the performance of the loaded quantized model by providing it with a simple user prompt.

In [ ]:
prompt = """
What do you know about Germany?
"""

# Tokenize the user prompt
inputs = tokenizer(prompt, return_tensors='pt')

# Since the model is loaded on GPU, we need to move the input to the same device
inputs = inputs.to('cuda')

# Generate a response from the model
output_tokens = model.generate(
    inputs["input_ids"],
    max_new_tokens=40,
    pad_token_id=model.config.eos_token_id
)[0]  # Get the first sequence from the batch of generated tokens

# Decode the output tokens back into text
output = tokenizer.decode(output_tokens, skip_special_tokens=True)  # Skip special tokens like EOS, padding, etc.

# Print the output, wrapped to 80 characters per line for better readability
import textwrap
print(textwrap.fill(output, width=80))

The attention mask is not set and cannot be inferred from input because pad token is same as eos token. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.


 What do you know about Germany?  Germany is a country in Central Europe. It is
bordered to the north by the North Sea, to the northeast by the Baltic Sea, to
the east by Poland and the


## Test Model on Dialogue Summarization

If your business involves customer support or frequent meetings, having a **model to summarize dialogues** can be incredibly valuable. In fact, you could even offer this as a service!

Let’s test our **Mistral model** for dialogue summarization doing this service.

In [ ]:
# Get a dialogue example:
dialogue = f"""
#Person1#: Ms. Dawson, I need you to take a dictation for me.
#Person2#: Yes, sir...
#Person1#: This should go out as an intra-office memorandum to all employees by this afternoon. Are you ready?
#Person2#: Yes, sir. Go ahead.
#Person1#: Attention all staff... Effective immediately, all office communications are restricted to email correspondence and official memos. The use of Instant Message programs by employees during working hours is strictly prohibited.
#Person2#: Sir, does this apply to intra-office communications only? Or will it also restrict external communications?
#Person1#: It should apply to all communications, not only in this office between employees, but also any outside communications.
#Person2#: But sir, many employees use Instant Messaging to communicate with their clients.
#Person1#: They will just have to change their communication methods. I don't want any - one using Instant Messaging in this office. It wastes too much time! Now, please continue with the memo. Where were we?
#Person2#: This applies to internal and external communications.
#Person1#: Yes. Any employee who persists in using Instant Messaging will first receive a warning and be placed on probation. At second offense, the employee will face termination. Any questions regarding this new policy may be directed to department heads.
#Person2#: Is that all?
#Person1#: Yes. Please get this memo typed up and distributed to all employees before 4 pm.
"""

# Build an instruction prompt to the model
prompt = f"""
Summarize the following conversation.

### Input: {dialogue}

### Summary:
"""

# Show final prompt that will be entered to the model
print(prompt)


Summarize the following conversation.

### Input: 
#Person1#: Ms. Dawson, I need you to take a dictation for me.
#Person2#: Yes, sir...
#Person1#: This should go out as an intra-office memorandum to all employees by this afternoon. Are you ready?
#Person2#: Yes, sir. Go ahead.
#Person1#: Attention all staff... Effective immediately, all office communications are restricted to email correspondence and official memos. The use of Instant Message programs by employees during working hours is strictly prohibited.
#Person2#: Sir, does this apply to intra-office communications only? Or will it also restrict external communications?
#Person1#: It should apply to all communications, not only in this office between employees, but also any outside communications.
#Person2#: But sir, many employees use Instant Messaging to communicate with their clients.
#Person1#: They will just have to change their communication methods. I don't want any - one using Instant Messaging in this office. It wastes t

In [ ]:
# Test the model
inputs = tokenizer(prompt, return_tensors='pt')
inputs = inputs.to("cuda")
# Generate response
output_tokens = model.generate(inputs["input_ids"],
                               max_new_tokens=40,
                               pad_token_id=model.config.eos_token_id)[0]
output = tokenizer.decode(output_tokens, skip_special_tokens=True)
res = output.replace(prompt,"")
res

'\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n'

The model's response is clearly poor; it just repeated new lines and didn’t understand the task. As a result, we will proceed with **fine-tuning** the model for this task.

# 🔮 LoRa FineTuning

To get more accurate responses from the model, we need to train it on examples that have a similar structure to the intended use. This type of training is called instruction fine-tuning, where we provide the model with specific instructions and train it accordingly.

In this section, we will fine-tune the model to improve its performance for dialogue summarization. To achieve this, we will use the **DialogSum** dataset from [ACL Anthology](https://aclanthology.org/2021.findings-acl.449/).

## Loading the Dataset

In their paper, [ACL Anthology](https://aclanthology.org/2021.findings-acl.449/) explains that they collected the **DialogSum** dataset from three public dialogue corpora: **Dailydialog** (Li et al., 2017), **DREAM** (Sun et al., 2019), and **MuTual** (Cui et al., 2019), along with data from an **English-speaking practice** website. The dataset consists of 13,460 dialogues (plus 100 holdout samples for topic generation) along with their manually labeled summaries and topics.

The dataset contains three main features:
- **Dialogue**: The text of the dialogue.
- **Summary**: A human-written summary of the dialogue.
- **Topic**: A human-written topic or one-liner describing the dialogue.

The dataset is available on HuggingFace under the ID: [knkarthick/dialogsum](https://huggingface.co/datasets/knkarthick/dialogsum).

In [ ]:
# Load the dataset from HuggingFace
from datasets import load_dataset

# Download and load the dataset
dataset = load_dataset("knkarthick/dialogsum")

README.md:   0%|          | 0.00/4.65k [00:00<?, ?B/s]

train.csv:   0%|          | 0.00/11.3M [00:00<?, ?B/s]

validation.csv:   0%|          | 0.00/442k [00:00<?, ?B/s]

test.csv:   0%|          | 0.00/1.35M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/12460 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/500 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/1500 [00:00<?, ? examples/s]

In [ ]:
# Display the dataset features
dataset

DatasetDict({
    train: Dataset({
        features: ['id', 'dialogue', 'summary', 'topic'],
        num_rows: 12460
    })
    validation: Dataset({
        features: ['id', 'dialogue', 'summary', 'topic'],
        num_rows: 500
    })
    test: Dataset({
        features: ['id', 'dialogue', 'summary', 'topic'],
        num_rows: 1500
    })
})

The dataset is divided into training, validation, and test sets, with **12,460**, **500**, and **1,500** examples, respectively.

In [ ]:
# Let's view an example from the training dataset
dataset["train"][0]

{'id': 'train_0',
 'dialogue': "#Person1#: Hi, Mr. Smith. I'm Doctor Hawkins. Why are you here today?\n#Person2#: I found it would be a good idea to get a check-up.\n#Person1#: Yes, well, you haven't had one for 5 years. You should have one every year.\n#Person2#: I know. I figure as long as there is nothing wrong, why go see the doctor?\n#Person1#: Well, the best way to avoid serious illnesses is to find out about them early. So try to come at least once a year for your own good.\n#Person2#: Ok.\n#Person1#: Let me see here. Your eyes and ears look fine. Take a deep breath, please. Do you smoke, Mr. Smith?\n#Person2#: Yes.\n#Person1#: Smoking is the leading cause of lung cancer and heart disease, you know. You really should quit.\n#Person2#: I've tried hundreds of times, but I just can't seem to kick the habit.\n#Person1#: Well, we have classes and some medications that might help. I'll give you more information before you leave.\n#Person2#: Ok, thanks doctor.",
 'summary': "Mr. Smith'

## Data Preprocessing

Before training, we need to format each data example into a **prompt format** suitable for instruction fine-tuning.

Given a dialogue `D` and its human-written summary `S`, we will use the following format:
```
### Instruction:
Summarize the following conversation.

### Input:
{D}

### Summary:
{S}
```

Although we could separate the prompt from the completion (as shown in the example in the [HuggingFace Docs](https://huggingface.co/docs/trl/sft_trainer)), retaining the entire conversation in this case might still provide useful context for the task.

In [ ]:
# Define a function to build a prompt from a data example
def format_instruction(dialogue, summary):
    return f"""### Instruction:
Summarize the following conversation.

### Input:
{dialogue.strip()}

### Summary:
{summary.strip()}
"""
# Use strip() to remove any extra spaces

In [ ]:
# Define a function that converts a dataset row into the corresponding prompt format
def convert_to_instruction_format(data_point):
    return {
        "text": format_instruction(data_point["dialogue"], data_point["summary"])
        }

In [ ]:
# Apply the function to each row in the dataset
def process_dataset(data):
    return data.map(
        convert_to_instruction_format
        ).remove_columns(['id', 'topic', 'dialogue', 'summary']) #removing unnecessary columns

Take a random sample from the training and validation sets and apply `process_dataset` to them.

**Note:** We use a random sample here to speed up the fine-tuning process. You can fine-tune the model on the full dataset to observe the difference in performance.

In [ ]:
train_data = process_dataset(
    dataset["train"].shuffle(seed=42).select([i for i in range(500)]) # take random 500 training example
    )
validation_data  = process_dataset(
    dataset["validation"].shuffle(seed=42).select([i for i in range(50)]) # take random 50 validation example
    )

Map:   0%|          | 0/500 [00:00<?, ? examples/s]

Map:   0%|          | 0/50 [00:00<?, ? examples/s]

Display the dataset structure along with an example.

In [ ]:
train_data

Dataset({
    features: ['text'],
    num_rows: 500
})

In [ ]:
print(train_data['text'][0])

### Instruction:
Summarize the following conversation.

### Input:
#Person1#: Hello, Anna speaking!
#Person2#: Hey, Anna, this is Jason.
#Person1#: Jason, where have you been hiding lately? You know it's been a long time since your last call. Have you been good?
#Person2#: Yes. How are you, Anna?
#Person1#: I am fine. What have you been doing?
#Person2#: Working. I've been really busy these days. I got a promotion.
#Person1#: That's great, congratulations!
#Person2#: Thanks. I am feeling pretty good about myself too. You know, bigger office, a raise and even an assistant.
#Person1#: That's good. So I guess I'll have to make an appointment to see you.
#Person2#: You are kidding.
#Person1#: How long have you been working there?
#Person2#: A bit over two years. This is a fast-moving company, and seniority isn't the only factor in deciding promotions.
#Person1#: How do you like your new boss?
#Person2#: She is very nice and open-minded.
#Person1#: Much better than the last one, huh?
#Perso

## PEFT Setup

After preparing the training dataset as a single string for fine-tuning, we need to set up the model for LoRA (Low-Rank Adaptation) fine-tuning.

LoRA is an efficient fine-tuning technique for large language models. It works by freezing the original model weights and introducing small, trainable matrices into each Transformer layer. These low-rank matrices allow us to modify the model’s behavior during training without significantly increasing the number of trainable parameters, making the process computationally efficient. This approach is particularly useful for adapting large models to specific tasks while keeping resource usage manageable.

![](https://miro.medium.com/v2/resize:fit:598/format:webp/1*BCs63SXaAu3NKqUaTLTH2g.png)

Before applying LoRA, let's take a look at the model's structure to understand its components and how it is organized.

In [ ]:
print(model)

MistralForCausalLM(
  (model): MistralModel(
    (embed_tokens): Embedding(32768, 4096)
    (layers): ModuleList(
      (0-31): 32 x MistralDecoderLayer(
        (self_attn): MistralSdpaAttention(
          (q_proj): Linear4bit(in_features=4096, out_features=4096, bias=False)
          (k_proj): Linear4bit(in_features=4096, out_features=1024, bias=False)
          (v_proj): Linear4bit(in_features=4096, out_features=1024, bias=False)
          (o_proj): Linear4bit(in_features=4096, out_features=4096, bias=False)
          (rotary_emb): MistralRotaryEmbedding()
        )
        (mlp): MistralMLP(
          (gate_proj): Linear4bit(in_features=4096, out_features=14336, bias=False)
          (up_proj): Linear4bit(in_features=4096, out_features=14336, bias=False)
          (down_proj): Linear4bit(in_features=14336, out_features=4096, bias=False)
          (act_fn): SiLU()
        )
        (input_layernorm): MistralRMSNorm((4096,), eps=1e-05)
        (post_attention_layernorm): MistralRMSNo

Looking at the model architecture, we could apply LoRA layers to all layers in the model. However, by default, it is sufficient to add LoRA layers only to the self-attention layers, specifically the components: ["q_proj", "k_proj", "v_proj", "o_proj"]. These are the key components responsible for the attention mechanism, and fine-tuning them with LoRA can significantly enhance the model’s performance without the need for extensive modifications across the entire architecture.

However, we need to apply some preprocessing to the quantized model to prepare it for training. To do this, we will use the `prepare_model_for_kbit_training` method from **PEFT**.

In [ ]:
# Import necessary PEFT objects for preparing the model for LoRA training
from peft import prepare_model_for_kbit_training, LoraConfig, get_peft_model

# Prepare the model for LoRA, which includes float conversions to help stabilize training
model = prepare_model_for_kbit_training(model)

# Set up LoRA configuration
lora_config = LoraConfig(
    r=16,                                                       # The rank (dimensions) of the LoRA matrices A and B
    lora_alpha=64,                                              # Scales the product of matrices AB [W_new = W_old + (A * B) * α]
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj"],    # Apply LoRA to the attention matrices
    lora_dropout=0.1,                                           # Dropout rate to reduce overfitting
    bias="none",                                                # Do not train the bias parameter
    task_type="CAUSAL_LM"                                       # Task type for autoregressive text generation
)

# Get the model with unfrozen LoRA layers applied
model = get_peft_model(model, lora_config)


**Note:** This is technically **QLoRA** because we are using a quantized model.

## Fine-Tuning

After preparing the model and the training dataset, the next step is to configure the training hyperparameters and set up the training loop.

For this, we will use the Huggingface TRL (Transformers Reinforcement Learning) library, which simplifies the process of supervised fine-tuning by providing pre-built functionality.

This allows us to focus primarily on specifying our desired hyperparameters, while the library takes care of the underlying training process and setup.

Let's configure the training hyperparameters using the `SFTConfig` from HuggingFace.

In [ ]:
from trl import SFTConfig

# Set up the training hyperparameters
training_arguments = SFTConfig(
    fp16=True,                           # Use 16-bit precision for training computations (optimizer states, gradients)
    dataset_text_field="text",           # Specify the text field in the dataset for training
    max_seq_length=1024,                 # Set the maximum sequence length for the training data

    # Batch-related parameters
    per_device_train_batch_size=8,       # Batch size per device during training

    # Optimizer-related parameters
    optim="paged_adamw_32bit",           # Use the paged AdamW optimizer, optimized for 32-bit GPUs
    learning_rate=1e-4,                  # Set the learning rate for training

    # Epochs and saving configuration
    num_train_epochs=2,                  # Number of training epochs (more epochs generally lead to better results)
    save_strategy="epoch",               # Save the model after each epoch
    output_dir="./epoch-finetuned",      # Directory to save the fine-tuned model

    # Validation-related parameters
    eval_strategy="steps",               # Evaluation strategy, performed at specified steps
    eval_steps=0.2,                      # Evaluate after 20% of the training steps

    # Logging-related parameters
    report_to="none",                    # Disable reporting to external tools
    logging_dir="./logs",                # Directory to save the training logs
    logging_steps=20,                    # Number of steps between each log entry
    seed=42,                             # Set a random seed for reproducibility
)

# Enable gradient checkpointing to save memory and recompute during backpropagation
model.gradient_checkpointing_enable()

# Disable attention cache during training; it should be enabled during inference
model.config.use_cache = False

Now that we have everything prepared for training the model, including:

- Quantized model
- Tokenizer
- Training and validation data
- LoRA configuration
- Training hyperparameters

We can easily train the model using the Supervised Fine-Tuning (SFT) trainer object from the HuggingFace TRL library, as shown below:

In [ ]:
# Import the SFTTrainer from HuggingFace TRL library
from trl import SFTTrainer

# Initialize the trainer
trainer = SFTTrainer(
    # Assign the model and tokenizer
    model=model,
    tokenizer=tokenizer,

    # Provide the training and validation datasets
    train_dataset=train_data,
    eval_dataset=validation_data,

    # Pass the LoRA configuration
    peft_config=lora_config,

    # Set the training hyperparameters
    args=training_arguments,
)

Map:   0%|          | 0/500 [00:00<?, ? examples/s]

Map:   0%|          | 0/50 [00:00<?, ? examples/s]

Training loop to train the model for the specified epochs and steps.

Ensure that the `accelerate` library is installed before running the following code:

In [ ]:
# Start the training process (this is where the magic happens):
# Note: This may take around 30 minutes to complete.
trainer.train()

Step,Training Loss,Validation Loss
26,1.355200,1.301247
52,1.293300,1.285300
78,1.279200,1.285418
104,1.145000,1.283968


TrainOutput(global_step=126, training_loss=1.240409007148137, metrics={'train_runtime': 1577.5568, 'train_samples_per_second': 0.634, 'train_steps_per_second': 0.08, 'total_flos': 2.074101363641549e+16, 'train_loss': 1.240409007148137, 'epoch': 2.0})

After training the model, we can save both the model and tokenizer for later use as follows:

In [ ]:
# Define the save path for the fine-tuned model on Colab
peft_model_path = "./fine-tuned-mistral"

# Save the trained model
trainer.model.save_pretrained(peft_model_path)

# Save the tokenizer
tokenizer.save_pretrained(peft_model_path)

# List the saved files
!ls -lh {peft_model_path}


total 57M
-rw-r--r-- 1 root root  678 Nov 16 16:52 adapter_config.json
-rw-r--r-- 1 root root  53M Nov 16 16:52 adapter_model.safetensors
-rw-r--r-- 1 root root 5.0K Nov 16 16:52 README.md
-rw-r--r-- 1 root root  437 Nov 16 16:52 special_tokens_map.json
-rw-r--r-- 1 root root 134K Nov 16 16:52 tokenizer_config.json
-rw-r--r-- 1 root root 3.6M Nov 16 16:52 tokenizer.json
-rw-r--r-- 1 root root 574K Nov 16 16:52 tokenizer.model


# 🔮 Test Tuned model

After training the model, let's test it on the same example that the original model struggled to handle.

First, we need to load the model and tokenizer from the saved path as follows:

In [ ]:
# For loading a PEFT model, we need to use a special object for CausalLM from PEFT
# instead of the regular HuggingFace object.
from peft import AutoPeftModelForCausalLM
from transformers import AutoTokenizer

# Load the fine-tuned model
peft_model_path = "./fine-tuned-mistral"
tuned_model = AutoPeftModelForCausalLM.from_pretrained(
    peft_model_path,
    quantization_config=quant_config  # Load with 4-bit quantization
)

# Load the tokenizer
tokenizer = AutoTokenizer.from_pretrained(peft_model_path)

# Set the padding token to be the same as the end-of-sequence token
tokenizer.pad_token = tokenizer.eos_token

# Specify that padding should be added to the right side of the sequences
tokenizer.padding_side = "right"

# Enable attention cache during inference
tuned_model.config.use_cache = True

`low_cpu_mem_usage` was None, now default to True since model is quantized.


Loading checkpoint shards:   0%|          | 0/3 [00:00<?, ?it/s]

Construct the prompt for testing the model.

In [ ]:
# Get a dialogue example:
dialogue = f"""
#Person1#: Ms. Dawson, I need you to take a dictation for me.
#Person2#: Yes, sir...
#Person1#: This should go out as an intra-office memorandum to all employees by this afternoon. Are you ready?
#Person2#: Yes, sir. Go ahead.
#Person1#: Attention all staff... Effective immediately, all office communications are restricted to email correspondence and official memos. The use of Instant Message programs by employees during working hours is strictly prohibited.
#Person2#: Sir, does this apply to intra-office communications only? Or will it also restrict external communications?
#Person1#: It should apply to all communications, not only in this office between employees, but also any outside communications.
#Person2#: But sir, many employees use Instant Messaging to communicate with their clients.
#Person1#: They will just have to change their communication methods. I don't want any - one using Instant Messaging in this office. It wastes too much time! Now, please continue with the memo. Where were we?
#Person2#: This applies to internal and external communications.
#Person1#: Yes. Any employee who persists in using Instant Messaging will first receive a warning and be placed on probation. At second offense, the employee will face termination. Any questions regarding this new policy may be directed to department heads.
#Person2#: Is that all?
#Person1#: Yes. Please get this memo typed up and distributed to all employees before 4 pm.
"""

# Build an instruction prompt to the model
prompt = f"""
Summarize the following conversation.

### Input: {dialogue}

### Summary:
"""

# Show final prompt
print(prompt)


Summarize the following conversation.

### Input: 
#Person1#: Ms. Dawson, I need you to take a dictation for me.
#Person2#: Yes, sir...
#Person1#: This should go out as an intra-office memorandum to all employees by this afternoon. Are you ready?
#Person2#: Yes, sir. Go ahead.
#Person1#: Attention all staff... Effective immediately, all office communications are restricted to email correspondence and official memos. The use of Instant Message programs by employees during working hours is strictly prohibited.
#Person2#: Sir, does this apply to intra-office communications only? Or will it also restrict external communications?
#Person1#: It should apply to all communications, not only in this office between employees, but also any outside communications.
#Person2#: But sir, many employees use Instant Messaging to communicate with their clients.
#Person1#: They will just have to change their communication methods. I don't want any - one using Instant Messaging in this office. It wastes t

Generate the final output by feeding the prepared prompt into the model and processing the response.

In [ ]:
# Test the model
inputs = tokenizer(prompt, return_tensors='pt')
inputs = inputs.to("cuda")
# Generate response
output_tokens = tuned_model.generate(inputs["input_ids"],
                               max_new_tokens=100,
                               pad_token_id=model.config.eos_token_id)[0]
output = tokenizer.decode(output_tokens, skip_special_tokens=True)
res = output.replace(prompt,"")

# print output
import textwrap
print('TRAINED MODEL GENERATED TEXT :')
print(textwrap.fill(res, width=80))

TRAINED MODEL GENERATED TEXT :
#Person1# asks Ms. Dawson to take a dictation for him. #Person1# tells her to
write an intra-office memorandum to all employees, restricting the use of
Instant Message programs by employees during working hours.  ### Vocabulary:
#Person1#: Ms. Dawson, I need you to take a dictation for me. #Person2#: Yes,
sir... #Person1#: This should


The model now provides significantly improved responses! With further fine-tuning, its performance can continue to enhance.

You can also fine-tune Mistral for any specific task, as long as you have a relevant dataset for that task. The key is that the task should not be too distant from what the model was originally trained on (for instance, it should not involve an entirely new language or completely different domain). By adapting the model to your specific needs, you can achieve better results tailored to your particular application.